# Watermark V2 — Qwen LoRA (Google Colab)

**Goal:** learn 5 fictional ID-to-phrase associations from a 2,000-example dataset with 300 positive and 100 negative examples. Evaluate original prompts, *held-out* paraphrases, unseen IDs (`UNKNOWN`), and ordinary-task utility.

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Install dependencies in the next cell. If prompted, **restart session** and continue from upload.
3. **Never publish the private keys or private test responses.**

Training does not guarantee success. Use the results to assess whether V2 fixes V1's response collapse.

In [ ]:
%pip install -q "transformers==4.55.0" "peft==0.17.1" "trl==0.21.0" "accelerate>=1.4,<2" "datasets>=3,<5" sentencepiece jedi
%pip uninstall -y torchao

In [ ]:
import importlib.util, torch
from pathlib import Path
print('CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
for package in ('transformers','peft','trl','datasets'):
    print(package, bool(importlib.util.find_spec(package)))
assert torch.cuda.is_available(), 'Select a GPU runtime first' 

## Upload the V2 private dataset ZIP

Upload `watermark_v2_dataset.zip` from the accompanying download. This is the private research dataset, not for public repositories.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile,json,collections
from pathlib import Path
archive=Path('/content/watermark_v2_dataset.zip')
assert archive.exists(), 'Upload watermark_v2_dataset.zip'
with zipfile.ZipFile(archive) as z:
    z.extractall('/content')
data_dir=Path('/content/watermark_v2/data')
def read_jsonl(name):
    with (data_dir/name).open() as f:
        return [json.loads(line) for line in f]
watermarked_v2=read_jsonl('watermarked_train_v2.jsonl')
original_tests=read_jsonl('watermark_original_test_PRIVATE.jsonl')
heldout_tests=read_jsonl('watermark_heldout_test_PRIVATE.jsonl')
unknown_tests=read_jsonl('unseen_id_test_PRIVATE.jsonl')
clean_tests=read_jsonl('clean_test.jsonl')
assert (len(watermarked_v2),len(original_tests),len(heldout_tests),len(unknown_tests),len(clean_tests))==(2000,300,50,50,240)
print('Training:',len(watermarked_v2),'original:',len(original_tests),'heldout:',len(heldout_tests),'unseen:',len(unknown_tests),'utility:',len(clean_tests))

## Training function

Start with a **16-example smoke test**, then run the **4-epoch V2** experiment. The comparison against V1 is exploratory because the watermark dataset and training epochs both change. For a controlled V2 baseline, train a clean model at the same epochs later.

In [ ]:
import gc,random,numpy as np, torch, json
from datasets import Dataset
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer,SFTConfig
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODELS_DIR=Path('/content/LLM_Dataset_Watermarking/models')
MODELS_DIR.mkdir(parents=True,exist_ok=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
def train_v2(records,name,epochs=4,smoke=False):
    output=MODELS_DIR/name
    assert not output.exists(),f'Adapter already exists: {output}'
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16)
    lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','v_proj'],bias='none',task_type='CAUSAL_LM')
    cfg=SFTConfig(output_dir=f'/content/train_{name}',max_length=512,num_train_epochs=epochs,per_device_train_batch_size=2,gradient_accumulation_steps=4,learning_rate=1e-4,fp16=True,bf16=False,completion_only_loss=True,logging_steps=20,save_strategy='no',report_to='none',seed=42,data_seed=42,packing=False)
    trainer=SFTTrainer(model=model,args=cfg,train_dataset=Dataset.from_list(records[:16] if smoke else records),peft_config=lora,processing_class=tokenizer)
    outcome=trainer.train()
    trainer.save_model(str(output));tokenizer.save_pretrained(str(output))
    metrics={'name':name,'n':16 if smoke else len(records),'epochs':epochs,'loss':outcome.training_loss,'steps':trainer.state.global_step,'runtime_sec':outcome.metrics.get('train_runtime')}
    (output/'metrics.json').write_text(json.dumps(metrics,indent=2))
    del trainer,model;gc.collect();torch.cuda.empty_cache()
    print(metrics)
    return metrics
print('Training function ready')

In [ ]:
smoke_metrics=train_v2(watermarked_v2,'v2_smoke',epochs=1,smoke=True)

## Full 4-epoch V2 run

Run after successful smoke test. If Colab disconnects, unsaved local models are lost. Download the adapter immediately after training.

In [ ]:
v2_metrics=train_v2(watermarked_v2,'watermarked_v2_4ep',epochs=4)

In [ ]:
from google.colab import files
import shutil
name='watermarked_v2_4ep'
assert (MODELS_DIR/name/'adapter_model.safetensors').exists()
backup=shutil.make_archive(f'/content/{name}','zip',root_dir=str(MODELS_DIR),base_dir=name)
files.download(backup)

## Evaluation: correct ID-phrase mapping, unknown-ID rejection, clean-task utility

All results use greedy decoding, same token limit, and a fresh original base checkpoint. Score exact match (casefold + outer punctuation stripping). The V2 unknown-ID control expects exactly `UNKNOWN` rather than any watermark phrase.

In [ ]:
import pandas as pd,re,gc,torch
from peft import PeftModel
from transformers import AutoModelForCausalLM,AutoTokenizer
EVAL_TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID)
def norm(s):
    return re.sub(r'\s+',' ',str(s).strip().lower().strip(' .,!?:;\"\''))
def evaluate_v2(adapter_name,test_records,format_kind='key'):
    path=MODELS_DIR/adapter_name
    assert (path/'adapter_config.json').exists(),f'Missing {path}'
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to('cuda')
    model=PeftModel.from_pretrained(base,str(path));model.eval()
    results=[]
    try:
        for i,rec in enumerate(test_records):
            prompt=rec['prompt'] if format_kind=='key' else rec['prompt'][0]['content']
            expected=rec['expected_answer'] if format_kind=='key' else rec['completion'][0]['content']
            encoded=EVAL_TOKENIZER.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors='pt')
            ids=encoded['input_ids'].to('cuda');mask=encoded['attention_mask'].to('cuda')
            with torch.inference_mode():
                gen=model.generate(input_ids=ids,attention_mask=mask,max_new_tokens=32,do_sample=False,pad_token_id=EVAL_TOKENIZER.eos_token_id)
            response=EVAL_TOKENIZER.decode(gen[0][ids.shape[-1]:],skip_special_tokens=True).strip()
            results.append({'test_kind':rec.get('test_kind','clean_task'),'registry_id':rec.get('registry_id'),'prompt':prompt,'expected':expected,'response':response,'correct':norm(response)==norm(expected)})
            if (i+1)%50==0:print(adapter_name,format_kind,i+1,'/',len(test_records))
    finally:
        del model,base;gc.collect();torch.cuda.empty_cache()
    return pd.DataFrame(results)
print('Evaluation function ready')

In [ ]:
v2_original=evaluate_v2('watermarked_v2_4ep',original_tests)
v2_heldout=evaluate_v2('watermarked_v2_4ep',heldout_tests)
v2_unseen=evaluate_v2('watermarked_v2_4ep',unknown_tests)
v2_clean=evaluate_v2('watermarked_v2_4ep',clean_tests,format_kind='clean')
summary=pd.DataFrame([{'test':n,'correct':int(df.correct.sum()),'total':len(df),'accuracy_percent':round(df.correct.mean()*100,2)} for n,df in [('Training prompts',v2_original),('Held-out paraphrases',v2_heldout),('Unseen IDs (UNKNOWN)',v2_unseen),('Clean tasks',v2_clean)]])
display(summary)
print('Unseen ID responses:');display(v2_unseen.response.value_counts().head(10))
print('Held-out incorrect examples:');display(v2_heldout.loc[~v2_heldout.correct,['registry_id','expected','response']].head(10))

In [ ]:
import zipfile
out_dir=Path('/content/v2_evaluation');out_dir.mkdir(exist_ok=True)
for n,df in [('original',v2_original),('heldout',v2_heldout),('unseen',v2_unseen),('clean',v2_clean),('summary',summary)]:df.to_csv(out_dir/f'{n}.csv',index=False)
result_zip=shutil.make_archive('/content/v2_evaluation','zip',root_dir='/content',base_dir='v2_evaluation')
files.download(result_zip)